# 📖 Notebook 9: Storage & Secrets — Persisting Data and Managing Configuration

Welcome back! In earlier notebooks, our FastAPI services mostly acted like stateless apps. In the real world, many workloads need configuration, passwords, and data that survives a restart. This notebook shows how Kubernetes handles those needs without baking everything into the container image.


## Learning Objectives

By the end of this notebook, you will be able to:

- Explain why containers are **ephemeral** and why that matters
- Use **ConfigMaps** for non-secret configuration
- Use **Secrets** for sensitive values such as usernames and passwords
- Explain why Kubernetes Secrets are base64-encoded but not encrypted by default
- Create and use a **PersistentVolumeClaim (PVC)**
- Describe the relationship between **PersistentVolumes (PV)**, **PersistentVolumeClaims (PVC)**, and **StorageClasses**
- Deploy a simple **StatefulSet** with persistent storage
- Understand the **External Secrets** pattern for production systems


## 🛠️ Setup

Before you run anything:

- Make sure you already completed the earlier Kubernetes notebooks
- Start Minikube if it is not running
- Run this lab from `03-technologies/container-orchestration/kubernetes/` so the sample manifests are easy to find
- Select the `.venv` kernel in VS Code's kernel picker (top-right of notebook). If it doesn't appear, reload the window: `Cmd+Shift+P` → 'Reload Window'.

We will use the sample microservices in the `k8s-lab` namespace:

- `api-gateway` on port `8000`
- `user-service` on port `8001`
- `order-service` on port `8002`


In [ ]:
!pwd
!kubectl config current-context
!kubectl get ns k8s-lab
!kubectl get pods -n k8s-lab
!kubectl get svc -n k8s-lab


## The Problem: Containers are Ephemeral

A container is designed to be disposable. Kubernetes can stop it, move it, or recreate it at any time. That is great for reliability, but it creates a new problem: **anything stored inside the container filesystem can disappear when the pod disappears**.

Think of a pod like a hotel room. You can leave your suitcase on the bed, but if the hotel closes your room and gives you a new one, your suitcase is gone unless you stored it somewhere permanent.

### ✅ Exercise
Create a pod, write a file inside it, delete the pod, then recreate it. You will see that the file does not come back.


In [ ]:
!kubectl delete pod ephemeral-demo -n k8s-lab --ignore-not-found
!kubectl run ephemeral-demo --image=busybox:1.36 -n k8s-lab --restart=Never --command -- sh -c "mkdir -p /demo && echo 'hello from an ephemeral pod' > /demo/demo.txt && sleep 3600"
!kubectl wait --for=condition=Ready pod/ephemeral-demo -n k8s-lab --timeout=120s
!kubectl exec -n k8s-lab ephemeral-demo -- cat /demo/demo.txt
!kubectl delete pod ephemeral-demo -n k8s-lab
!kubectl run ephemeral-demo --image=busybox:1.36 -n k8s-lab --restart=Never --command -- sh -c "mkdir -p /demo && sleep 3600"
!kubectl wait --for=condition=Ready pod/ephemeral-demo -n k8s-lab --timeout=120s
!kubectl exec -n k8s-lab ephemeral-demo -- ls /demo


### ASCII Diagram: How Persistent Storage Fits Together

```
┌──────────────┐
│     Pod      │
│ writes files │
└──────┬───────┘
       │ mounts
       ▼
┌──────────────┐
│     PVC      │  PersistentVolumeClaim
│ "I need 1Gi" │
└──────┬───────┘
       │ bound to
       ▼
┌──────────────┐
│      PV      │  PersistentVolume
│ actual disk  │
└──────┬───────┘
       │ backed by
       ▼
┌──────────────────────┐
│ Physical Storage     │
│ disk / SSD / cloud   │
└──────────────────────┘
```

The PVC is the request. The PV is the actual storage. In many clusters, a StorageClass automatically creates the PV for you when you create the PVC.


## 🧾 ConfigMaps

A **ConfigMap** stores non-secret configuration as key-value pairs. This is a good place for things like log level, environment name, feature flags, and cache settings.

Why is this useful? Because your container image stays the same, while your configuration can change per environment. The same image can run in dev, staging, and production with different settings.

### ✅ Exercise
Create a ConfigMap from literal values right from the command line.


In [ ]:
!kubectl delete configmap app-config -n k8s-lab --ignore-not-found
!kubectl create configmap app-config --from-literal=LOG_LEVEL=info --from-literal=ENV=dev -n k8s-lab
!kubectl get configmap app-config -n k8s-lab -o yaml


## Using a ConfigMap in a Pod

Kubernetes lets a pod consume ConfigMap data in two very common ways:

1. **As environment variables** — easy for apps that already read environment variables
2. **As mounted files in a volume** — useful when an app expects config files

We will do both in the same pod so you can compare the behavior.

### ✅ Exercise
Write a pod manifest that reads the same ConfigMap both as env vars and as files under `/config`.


In [ ]:
%%writefile config-demo-pod.yaml
apiVersion: v1
kind: Pod
metadata:
  name: config-demo
  namespace: k8s-lab
spec:
  containers:
    - name: app
      image: busybox:1.36
      command: ["sh", "-c", "sleep 3600"]
      env:
        - name: LOG_LEVEL
          valueFrom:
            configMapKeyRef:
              name: app-config
              key: LOG_LEVEL
        - name: ENV
          valueFrom:
            configMapKeyRef:
              name: app-config
              key: ENV
      volumeMounts:
        - name: config-volume
          mountPath: /config
  volumes:
    - name: config-volume
      configMap:
        name: app-config


In [ ]:
!kubectl delete pod config-demo -n k8s-lab --ignore-not-found
!kubectl apply -f config-demo-pod.yaml
!kubectl wait --for=condition=Ready pod/config-demo -n k8s-lab --timeout=120s
!kubectl exec -n k8s-lab config-demo -- printenv LOG_LEVEL ENV
!kubectl exec -n k8s-lab config-demo -- sh -c "ls /config && echo '---' && cat /config/LOG_LEVEL && echo && cat /config/ENV"


## Updating a ConfigMap

This is one of the most important behaviors to understand:

- **Environment variables do not change in a running pod**
- **Mounted ConfigMap files can update automatically** after Kubernetes refreshes the volume

In other words, environment variables are copied into the pod when it starts. Mounted files stay connected to the ConfigMap.

### ✅ Exercise
Update the ConfigMap and compare the env var with the mounted file. Then restart the pod to refresh the env var.


In [ ]:
!kubectl create configmap app-config --from-literal=LOG_LEVEL=debug --from-literal=ENV=prod -n k8s-lab -o yaml --dry-run=client | kubectl apply -f -
!kubectl exec -n k8s-lab config-demo -- printenv LOG_LEVEL ENV
!kubectl exec -n k8s-lab config-demo -- sh -c "echo 'Mounted files may take a short moment to refresh'; ls /config; echo '---'; cat /config/LOG_LEVEL; echo; cat /config/ENV"


In [ ]:
!kubectl delete pod config-demo -n k8s-lab
!kubectl apply -f config-demo-pod.yaml
!kubectl wait --for=condition=Ready pod/config-demo -n k8s-lab --timeout=120s
!kubectl exec -n k8s-lab config-demo -- printenv LOG_LEVEL ENV


## 🔐 Secrets

A **Secret** is similar to a ConfigMap, but it is meant for sensitive data such as passwords, tokens, and API keys. Kubernetes treats Secrets differently in tooling and APIs, but a Secret is still just a Kubernetes object stored in the cluster.

### ✅ Exercise
Create a Secret for database credentials.


In [ ]:
!kubectl delete secret db-creds -n k8s-lab --ignore-not-found
!kubectl create secret generic db-creds --from-literal=DB_USER=admin --from-literal=DB_PASS=secret123 -n k8s-lab
!kubectl get secret db-creds -n k8s-lab -o yaml


## Base64 Encoding vs Encryption

This is a very common beginner misunderstanding, so let's make it simple:

- **Base64 encoding** changes the format of data so it is safe to move around as text
- **Encryption** scrambles the data so only someone with the right key can read it

Base64 is like writing a message in a different alphabet that anyone can translate back. Encryption is like locking the message in a safe.

By default, Kubernetes Secrets are **base64-encoded, not encrypted**. That means you should still protect access to the cluster, use RBAC, and ideally enable encryption at rest or use an external secret store.

### ✅ Exercise
Decode the stored values and prove to yourself that base64 is reversible.


In [ ]:
!kubectl get secret db-creds -n k8s-lab -o jsonpath='{.data.DB_USER}' | base64 --decode && echo
!kubectl get secret db-creds -n k8s-lab -o jsonpath='{.data.DB_PASS}' | base64 --decode && echo


## Mounting Secrets as Environment Variables

Just like ConfigMaps, Secrets can be injected into a pod. One very common pattern is to expose secret values as environment variables that the app reads on startup.

### ✅ Exercise
Create a pod that reads `DB_USER` and `DB_PASS` from the Secret.


In [ ]:
%%writefile secret-demo-pod.yaml
apiVersion: v1
kind: Pod
metadata:
  name: secret-demo
  namespace: k8s-lab
spec:
  containers:
    - name: app
      image: busybox:1.36
      command: ["sh", "-c", "sleep 3600"]
      env:
        - name: DB_USER
          valueFrom:
            secretKeyRef:
              name: db-creds
              key: DB_USER
        - name: DB_PASS
          valueFrom:
            secretKeyRef:
              name: db-creds
              key: DB_PASS


In [ ]:
!kubectl delete pod secret-demo -n k8s-lab --ignore-not-found
!kubectl apply -f secret-demo-pod.yaml
!kubectl wait --for=condition=Ready pod/secret-demo -n k8s-lab --timeout=120s
!kubectl exec -n k8s-lab secret-demo -- printenv DB_USER DB_PASS


## 💾 PersistentVolumes and PersistentVolumeClaims

Now we solve the big storage problem. Instead of writing data only inside the pod filesystem, we attach storage through a **PersistentVolumeClaim**.

In Minikube, there is usually a default **StorageClass**. That means when you create a PVC, Kubernetes can dynamically create the backing storage for you.

### ✅ Exercise
Inspect the storage classes, then create a PVC for 1Gi of storage.


In [ ]:
!kubectl get storageclass


In [ ]:
%%writefile storage-demo-pvc.yaml
apiVersion: v1
kind: PersistentVolumeClaim
metadata:
  name: storage-demo-pvc
  namespace: k8s-lab
spec:
  accessModes:
    - ReadWriteOnce
  resources:
    requests:
      storage: 1Gi


In [ ]:
!kubectl apply -f storage-demo-pvc.yaml
!kubectl get pvc storage-demo-pvc -n k8s-lab
!kubectl get pv


## Proving That Data Persists

A PVC only becomes meaningful when a pod mounts it. We will create a pod that writes data into `/data`, delete the pod, recreate it, and then confirm that the file is still there.

### ✅ Exercise
Mount the PVC in a pod and test data persistence.


In [ ]:
%%writefile pvc-writer-pod.yaml
apiVersion: v1
kind: Pod
metadata:
  name: pvc-writer
  namespace: k8s-lab
spec:
  containers:
    - name: app
      image: busybox:1.36
      command: ["sh", "-c", "echo 'pod started' >> /data/history.txt && sleep 3600"]
      volumeMounts:
        - name: app-data
          mountPath: /data
  volumes:
    - name: app-data
      persistentVolumeClaim:
        claimName: storage-demo-pvc


In [ ]:
!kubectl delete pod pvc-writer -n k8s-lab --ignore-not-found
!kubectl apply -f pvc-writer-pod.yaml
!kubectl wait --for=condition=Ready pod/pvc-writer -n k8s-lab --timeout=120s
!kubectl exec -n k8s-lab pvc-writer -- sh -c "echo 'this line survives pod deletion' >> /data/history.txt && cat /data/history.txt"
!kubectl delete pod pvc-writer -n k8s-lab
!kubectl apply -f pvc-writer-pod.yaml
!kubectl wait --for=condition=Ready pod/pvc-writer -n k8s-lab --timeout=120s
!kubectl exec -n k8s-lab pvc-writer -- cat /data/history.txt


## StorageClass Basics

A **StorageClass** tells Kubernetes *how* to provision storage. Think of it like a template for disk creation.

Important ideas:

- **Provisioner**: the plugin or driver that actually creates storage
- **Reclaim policy**: what happens to the backing storage after the claim is deleted
  - `Delete` usually removes the underlying storage too
  - `Retain` keeps the storage around for manual recovery
- **Binding mode**: when the actual volume gets created and attached

### ✅ Exercise
Inspect the default StorageClass and look for its provisioner and reclaim policy.


In [ ]:
!kubectl get storageclass -o wide
!kubectl describe storageclass $(kubectl get storageclass -o jsonpath='{.items[0].metadata.name}')


## 🧠 StatefulSet: When Pod Identity Matters

A Deployment is great for stateless apps because any replica can replace any other replica. But databases and queues often need a stable identity and stable storage. That is where **StatefulSet** comes in.

A StatefulSet gives pods predictable names like `redis-0`, `redis-1`, and `redis-2`. Each pod can get its own volume.

### ✅ Exercise
Deploy a tiny Redis StatefulSet with persistent storage, set a key, restart the pod, and confirm the value still exists.


In [ ]:
%%writefile redis-statefulset.yaml
apiVersion: v1
kind: Service
metadata:
  name: redis
  namespace: k8s-lab
spec:
  clusterIP: None
  selector:
    app: redis
  ports:
    - port: 6379
      targetPort: 6379
---
apiVersion: apps/v1
kind: StatefulSet
metadata:
  name: redis
  namespace: k8s-lab
spec:
  serviceName: redis
  replicas: 1
  selector:
    matchLabels:
      app: redis
  template:
    metadata:
      labels:
        app: redis
    spec:
      containers:
        - name: redis
          image: redis:7-alpine
          command: ["redis-server", "--appendonly", "yes"]
          ports:
            - containerPort: 6379
          volumeMounts:
            - name: redis-data
              mountPath: /data
  volumeClaimTemplates:
    - metadata:
        name: redis-data
      spec:
        accessModes: ["ReadWriteOnce"]
        resources:
          requests:
            storage: 1Gi


In [ ]:
!kubectl apply -f redis-statefulset.yaml
!kubectl rollout status statefulset/redis -n k8s-lab --timeout=180s
!kubectl get pods,pvc -n k8s-lab | grep redis
!kubectl exec -n k8s-lab redis-0 -- redis-cli SET notebook9 stored
!kubectl exec -n k8s-lab redis-0 -- redis-cli GET notebook9
!kubectl delete pod redis-0 -n k8s-lab
!kubectl wait --for=condition=Ready pod/redis-0 -n k8s-lab --timeout=180s
!kubectl exec -n k8s-lab redis-0 -- redis-cli GET notebook9


## 🌐 External Secrets Pattern

In production, teams often avoid storing sensitive values directly in Kubernetes. Instead, they use an external secret store such as HashiCorp Vault, AWS Secrets Manager, Azure Key Vault, or Google Secret Manager.

The flow usually looks like this:

```
┌────────────────────┐
│ ExternalSecret CRD │
└─────────┬──────────┘
          │ read rule
          ▼
┌────────────────────┐
│ External provider  │  Vault / AWS / Azure / GCP
└─────────┬──────────┘
          │ sync value
          ▼
┌────────────────────┐
│ Kubernetes Secret  │
└─────────┬──────────┘
          │ consumed by
          ▼
┌────────────────────┐
│ Pod / Deployment   │
└────────────────────┘
```

This notebook does **not** require a Vault install. We will just look at what the CRD usually looks like.

### ✅ Exercise
Write an example `ExternalSecret` manifest and read through the fields.


In [ ]:
%%writefile external-secret-example.yaml
apiVersion: external-secrets.io/v1beta1
kind: ExternalSecret
metadata:
  name: db-creds-sync
  namespace: k8s-lab
spec:
  refreshInterval: 1h
  secretStoreRef:
    name: lab-secret-store
    kind: SecretStore
  target:
    name: db-creds
  data:
    - secretKey: DB_USER
      remoteRef:
        key: /k8s-lab/database
        property: username
    - secretKey: DB_PASS
      remoteRef:
        key: /k8s-lab/database
        property: password


In [ ]:
!cat external-secret-example.yaml


## ✅ Best Practices

Here are the habits you want in real projects:

- Do **not** commit real secrets to Git
- Use **ConfigMaps** for non-sensitive configuration only
- Use **Secrets** for sensitive values, but remember they need RBAC and platform protection too
- Enable **encryption at rest** for cluster data if your platform supports it
- Prefer **external secret operators** in production
- Choose the right **StorageClass** for the workload
- Use **StatefulSets** when identity and storage must stay attached to a workload


## 🧹 Clean Up

Run this when you want to remove the resources created by this notebook.


In [ ]:
!kubectl delete pod ephemeral-demo config-demo secret-demo pvc-writer -n k8s-lab --ignore-not-found
!kubectl delete secret db-creds -n k8s-lab --ignore-not-found
!kubectl delete configmap app-config -n k8s-lab --ignore-not-found
!kubectl delete pvc storage-demo-pvc -n k8s-lab --ignore-not-found
!kubectl delete -f redis-statefulset.yaml --ignore-not-found
!rm -f config-demo-pod.yaml secret-demo-pod.yaml storage-demo-pvc.yaml pvc-writer-pod.yaml redis-statefulset.yaml external-secret-example.yaml


## 🎓 What You Learned

Nice work. In this notebook you learned how Kubernetes handles the two things stateless demos usually skip: **configuration** and **data persistence**.

You practiced how to:

- Store non-secret settings in a **ConfigMap**
- Store sensitive values in a **Secret**
- Explain why base64 is not the same as encryption
- Request durable storage with a **PVC**
- See how a **StatefulSet** keeps identity and storage aligned
- Understand why production teams often use **external secret managers**

In the final notebook, we will zoom out and look at production-ready platform patterns: autoscaling, multi-tenancy, disruption budgets, and backup strategy.
